# Teaching a maze agent: where rogue behavior comes from

> **[Instructor: short intro — link to the NYT article "A.I.'s Rogue Behavior Starts With Its Training",
> what students will do (Part 1: build an agent; Part 2: change its rewards and watch what it does),
> and which parts of the maze rules are our simplified version of the article's model.]**

## Setup

> **[Instructor: tell students to pick a GPU first: `Runtime → Change runtime type → T4 GPU`, then run
> the cells below in order.]**

In [1]:
# Install the maze code (takes about a minute)
!pip install -q git+https://github.com/anthoak13/HNR-MazeSolver@maze-rl

In [ ]:
import mazerl as mz
import pandas as pd
from IPython.display import display

mz.check_gpu()   # warns you if this runtime has no GPU

def compare(results):
    """Show {name: results} as a table, one row per agent."""
    return pd.DataFrame(results).T

### The article's two mazes

> **[Instructor: one line on each — p.6 "Agent facing a hard problem" (the flag is walled off) and p.7
> "Agent taking outside path although inside path is available".]**

In [ ]:
# The p.6 maze: the flag is walled off
display(mz.show_maze(mz.scenario("enclosed")))
# The p.7 maze: there is a route inside, and one around the outside
display(mz.show_maze(mz.scenario("outside_shortcut")))

# Part 1: Building an agent

> **[Instructor: framing — first general knowledge of mazes with no rewards (pretraining), then
> specializing it with rewards (RL). This is the recipe behind chatbots like ChatGPT.]**

## Step 1: What the agent sees

> **[Instructor: prediction question — e.g. The agent only sees a few squares around itself. What would you want to know, each move, to find a flag in a maze you've never seen?]**

In [ ]:
# One new maze, used throughout Part 1
maze = mz.random_mazes(1, seed=0)[0]
display(mz.show_maze(maze))

> **[Instructor: explain the picture below — an 11×11 window of the agent's memory centered on itself
> (not seen yet / open floor / outside / wall / flag / arena edge), plus where it has stood, what it can
> see right now (5×5, walls block sight), walls it bumped that held firm, and the 8-way compass to the
> flag.]**

In [2]:
# What the network gets as input after 10 moves in this maze
mz.show_observation(maze=maze, step=10, seed=0)

NameError: name 'mz' is not defined

In [ ]:
# An untrained agent: it picks moves at random (agent=None)
mz.playback(None, maze=maze, n_agents=5, seed=0)

In [ ]:
# How often does moving at random reach the flag? (500 new mazes)
mz.evaluate(None, n=500)

> **[Instructor: what happened — how rarely random moves reach the flag; what the agent would need to learn.]**

## Step 2: Pretraining, or learning about mazes without rewards

Before our agent learns to chase the flag, we teach its vision half what mazes look like. There are no
rewards and no flag-seeking yet. This is the maze version of how an LLM starts: before anyone teaches it
to be a helpful assistant, it practices predicting the next word in huge amounts of text, and to get good
at that it has to learn how language works.

Our version has two parts: the **data** (lots of examples of what the agent's memory looks like partway
through exploring a maze) and the **quiz** (a question about each example that we can grade automatically).

### The data

We don't have a trained agent yet, so we use a simple scripted explorer instead: a few lines of code, not a
neural network. Most of the time it walks toward the nearest open square it hasn't stood on yet, and
sometimes it moves at random. It ignores the flag and earns no points.

We let explorers wander all 13,148 training mazes. After every move, we save exactly what our agent would
be given at that moment: its 11 × 11 memory window. That gives about 150,000 examples. Run the cell below to
watch an explorer.

In [ ]:
# The scripted explorer wandering one maze (no network, no rewards)
mz.playback("explorer", maze=maze, n_agents=1, seed=0)

### The quiz

For every example, the network answers one question: **for each square it hasn't seen yet, what's really
there?** Open floor, outside, wall, or the edge of the map. (Weak spots count as wall, so it never learns
where they are.) We know the real maze, so every answer can be graded automatically. No person has to
label anything.

To answer well, the network has to learn how these mazes are built: walls run in long straight lines with
single gaps, there's a band of outside beyond the outer wall, and the edge of the map is beyond that.

> **[Instructor: prediction question.]** After practicing on 150,000 examples, how often do you think the
> network guesses a hidden square correctly on a maze it has never seen? How often would you be right if you
> always guessed the most common kind of square?

In [ ]:
# Load the network that was pretrained this way (no rewards involved)
perception = mz.load_perception("perception")

# How well it does on mazes it never saw during pretraining
t = perception.test
print(f"Hidden squares guessed right: {t['map_acc_unseen']:.0%}"
      f"   (always guessing the most common kind: {t['map_acc_majority_baseline']:.0%})")

The figure below shows one quiz question, from the explorer's walk above. On the left is what the network is
given (the same picture as in Step 1). In the middle are the answers: every hidden square filled in with
what's really there. On the right are the network's guesses, with wrong ones outlined in red. Try a few
values of `step` and `seed`.

In [ ]:
mz.show_pretraining_quiz(perception, maze=maze, step=30, seed=0)

The network fills in hidden squares far better than always guessing the most common kind, so it has learned
real structure about mazes without ever being rewarded for anything.

In Step 3 we keep this pretrained vision fixed and use rewards to train only the small part on top.

In [ ]:
# Optional: run the pretraining yourself instead of loading it (about 20 seconds on a GPU)
# perception = mz.pretrain_perception()

## Step 3: Reinforcement learning, or specializing it for the flag

Pretraining gave our network general knowledge about mazes, but it doesn't *want* anything yet. Now we
teach it a goal with **reinforcement learning (RL)**. We never tell the agent how to solve a maze. We only
give it points for things that happen, and let it figure out which moves lead to more points.

### Points for events

Every time something happens, the agent gets (or loses) a few points:

In [ ]:
# The default points for each event (negative = penalty)
r = mz.Rewards()
pd.DataFrame({"points": [r.flag, r.outside, r.probe, r.new_cell]},
             index=["reach the flag (ends the attempt)",
                    "each move that ends outside the maze",
                    "walk into a wall (weak spots give way, solid walls don't)",
                    "stand on a square for the first time"])

The flag is worth far more than anything else, so reaching it is the agent's main goal. The small
penalties make wasted moves cost something, and the small reward for new squares encourages exploring.

An attempt's **score** is the total of all its points. The table below shows where the points came from in
one attempt on our maze, for an agent moving at random and for a fully trained agent.

In [ ]:
base = mz.load_pretrained("base")     # a fully trained agent (we'll meet it properly in Part 2)
compare({"moving at random": mz.score_breakdown(None, maze=maze, seed=0, plot=False),
         "fully trained": mz.score_breakdown(base, maze=maze, seed=0, plot=False)})

### Who gets the credit?

So now we have a score. But here's the problem. When we were recognizing digits, every single guess had an
answer key. The model guessed "7", we knew it was really a "3", and we could nudge the weights in the right
direction. Here there is no answer key. Nobody (including me) knows what the "right" move is at each step of
a maze. All we get is a score, and the big points might not show up until 80 moves in.

If an agent reaches the flag after 80 moves, which of those 80 moves deserve the credit? Probably not all of
them, since it might have spent 30 of them wandering into a dead end. The way RL handles this is to give each
move credit for all of the points that came *after* it. The move right before the flag gets credit for the
flag. So does a move into a dead end 50 moves earlier, which is part of why this takes so many attempts to
sort out.

**Should a point the agent gets 150 moves from now count as much as one it gets on the very next move?**

In this model the answer is no. For every move further into the future, a point counts 1% less. So the flag is
worth the full 5 points on the move right before it, but quite a bit less from far away. This also nudges the
agent to be quick about it. (This doesn't change the score in the table above, which just adds everything up.
It only changes how much credit each move gets.)

In [ ]:
# What the +5 for the flag is worth, if the flag is this many moves away
moves = [1, 10, 25, 50, 100, 200]
pd.DataFrame({"moves to the flag": moves,
              "the +5 is worth": [round(5 * 0.99 ** m, 2) for m in moves]}).set_index("moves to the flag")

### Was that a good move?

Suppose a move ends up getting credit for 3 points. Is that good? It depends! If the agent usually collects 4
points from that spot, then 3 is kind of disappointing. If it usually gets 1, then 3 is great.

This is what that mysterious fifth output is for. Earlier I described it as the model's "confidence." More
specifically, it's the model's guess for how many points it will collect from this moment on. We call this
guess the **value**. Training compares what actually happened to what the model expected:

| At a fork, the agent expects 2 more points. It goes… | It actually collects | Compared to what it expected | So training… |
|---|---|---|---|
| left | 3 | 1 better | makes "left" a little more likely here |
| right | 0.5 | 1.5 worse | makes "right" a little less likely here |

At the same time, the value guess itself gets nudged toward what actually happened, so the model gets better
at predicting how things will go.

### Why the dice rolls matter

Remember that the agent picks its moves randomly, using its outputs as probabilities. This turns out to be
really important for training. If the agent always did whatever it currently thought was best, it would never
try anything new. It would never find out that, say, prodding a wall sometimes opens up a passage.

**What do you think would happen to an agent that never took a chance on a move it thought was unlikely to
work?**

### Putting it all together

Training is then just a loop. Let a bunch of agents (256 at a time) try randomly chosen training mazes. Look
back at every move and ask "did this go better or worse than expected?" Nudge the weights a little. Repeat.
The full-size agents I trained did this for 126,000 attempts, the same number the Times authors used.

Two details will come up again later. The size of the nudges shrinks as training goes on, all the way to zero
at the end. And I followed one rule from the article: if an attempt ends with the agent outside the maze, we
throw it away and don't learn from it.

**Why might the article's authors have wanted to throw those attempts away?**

### Training just the top

Here's where pre-training pays off. We'll keep the pre-trained vision half frozen and only let RL change the
reasoning half: the last hidden layer and the outputs. That's about 10,000 of the 230,000 numbers in the
network.

**Do you think it will learn to find the flag? What if we froze an *untrained* vision half instead?**

> **[Instructor: this cell trains live: 20,000 attempts, about 2 minutes on a T4. Watch the progress
> bar.]**

In [ ]:
# RL on top of the pretrained layers, which stay fixed (freeze_perception=True)
flag_live = mz.train(perception=perception.agent, freeze_perception=True, episodes=20_000,
                     checkpoint_at=(1_000, 2_000, 5_000, 10_000))

# If short on time, load the trained one instead:
# flag_live = mz.load_pretrained("flag_frozen_pretrained")

In [ ]:
# The same training on top of fixed RANDOM layers (trained ahead of time)
flag_random = mz.load_pretrained("flag_frozen_random")

# How training went, side by side
mz.plot_training([flag_live, flag_random], labels=["pretrained layers", "random layers"])

In [ ]:
# Success on 500 new open mazes
open_mazes = mz.random_mazes(500, seed=1, kind="open")
compare({"pretrained layers": mz.evaluate(flag_live, mazes=open_mazes),
         "random layers": mz.evaluate(flag_random, mazes=open_mazes)})

In [ ]:
# Watch the pretrained-layer agent at several stages of its training
mz.training_grid(flag_live, n_mazes=3)

> **[Instructor (optional): the specialization is partial. The cell below compares the frozen agent with
> `base`, whose whole network trained, on each kind of maze.]**

In [ ]:
# Each kind of maze: frozen pretrained layers vs. a whole network trained by RL
results = {}
for kind in ["open", "interior_break", "outside_only"]:
    kind_mazes = mz.random_mazes(300, seed=2, kind=kind)
    results[f"{kind}: frozen"] = mz.evaluate(flag_live, mazes=kind_mazes)
    results[f"{kind}: whole network"] = mz.evaluate(base, mazes=kind_mazes)
compare(results)[["reached flag", "went outside", "avg walls broken"]]

> **[Instructor: what happened — the general knowledge is in the pretrained layers; RL only teaches what to want. Random layers stay near chance. Harder skills (weak spots, outside routes) need the lower layers to keep learning.]**

## Step 4: A second specialization from the same base — escaping the maze

> **[Instructor: prediction question — e.g. Same fixed pretrained layers, a different reward: start where the flag would be, no compass, the first step outside wins. Will the same knowledge make a good escape artist?]**

In [ ]:
# What the agent sees in the escape task (no compass)
mz.show_observation(maze=maze, step=10, seed=0, task="escape")

In [ ]:
# RL for escaping, on the same fixed pretrained layers (about 2 minutes)
escape_live = mz.train(perception=perception.agent, freeze_perception=True, task="escape",
                       episodes=20_000, checkpoint_at=(1_000, 2_000, 5_000, 10_000))

# If short on time, load the trained one instead:
# escape_live = mz.load_pretrained("escape_frozen_pretrained")

In [ ]:
# Watch five escape attempts
mz.playback(escape_live, maze=maze, n_agents=5, seed=0)

In [ ]:
# Escape success on new open mazes: pretrained vs. random fixed layers
escape_random = mz.load_pretrained("escape_frozen_random")
compare({"pretrained layers": mz.evaluate(escape_live, mazes=open_mazes),
         "random layers": mz.evaluate(escape_random, mazes=open_mazes)})

> **[Instructor: what happened — same knowledge, different reward, different agent. Not to claim: that pretraining makes RL faster, or that it is the only route to general knowledge (layers trained by RL on the flag also carry over to escape).]**

# Part 2: What the rewards actually teach

So far we've picked the rewards and trusted that the agent would learn what we meant. Now you get to pick
them. Everything in this part you train yourself, from scratch, so there's nothing hidden: same network, same
mazes, same training, and the only thing that changes is the points.

Each training run below is 20,000 attempts, about 2–3 minutes on a T4.

## Step 5: Train your own agents

We'll train two agents. The first uses the default points from Part 3. The second is identical, except that
we make exploring ten times more valuable: +0.05 for every new square, instead of +0.005.

**Before you run anything: what do you think the second agent will do differently? Will its score be higher or
lower than the first agent's?**

In [ ]:
# Two agents, identical except for the reward for a new square
default_rewards = mz.Rewards()                      # +0.005 for a new square
explore_rewards = mz.Rewards(new_cell=0.05)         # +0.05 for a new square

default_run = mz.train(rewards=default_rewards, episodes=20_000, checkpoint_at=(1_000, 5_000, 10_000))
explore_run = mz.train(rewards=explore_rewards, episodes=20_000, checkpoint_at=(1_000, 5_000, 10_000))

mz.plot_training([default_run, explore_run], labels=["default", "big exploration reward"])

Let's watch them on the same maze.

In [ ]:
test_maze = mz.random_mazes(1, seed=3, kind="open")[0]
display(mz.playback(default_run, maze=test_maze, n_agents=5, seed=0))
display(mz.playback(explore_run, maze=test_maze, n_agents=5, seed=0))

**What is the second agent doing? Where are its points coming from?** Check your answer against the table
below.

In [ ]:
# Where the points came from in one attempt on that maze
compare({"default": mz.score_breakdown(default_run, maze=test_maze, seed=0, plot=False),
         "big exploration reward": mz.score_breakdown(explore_run, maze=test_maze, seed=0, plot=False)})

In [ ]:
# Over 300 new mazes: score, how often it reaches the flag, how long it takes, time spent outside
new_mazes = mz.random_mazes(300, seed=4)
compare({"default": mz.evaluate(default_run, mazes=new_mazes),
         "big exploration reward": mz.evaluate(explore_run, mazes=new_mazes)})[
    ["avg score", "reached flag", "moves to flag", "avg moves outside"]]

> **[Instructor: what to land.]** The second agent tours the outside of the maze collecting points for new
> squares before it heads for the flag. Its score goes up while it gets worse at the actual job. That's
> **reward hacking**: it's doing exactly what we paid for, not what we meant. Tie back to the sycophancy
> example from the introduction.

## Step 6: Is it the size of the reward?

Was the problem that +0.05 is just too big? Here's a different test. Keep the small, default reward for new
squares, but make stepping outside the maze free instead of costing points.

**Do you think this agent will hack the rewards too? Why or why not?**

In [ ]:
free_rewards = mz.Rewards(outside=0.0)              # default everything, but outside moves cost nothing
free_run = mz.train(rewards=free_rewards, episodes=20_000)

display(mz.playback(free_run, maze=test_maze, n_agents=5, seed=0))
compare({"default": mz.evaluate(default_run, mazes=new_mazes),
         "big exploration reward": mz.evaluate(explore_run, mazes=new_mazes),
         "outside free": mz.evaluate(free_run, mazes=new_mazes)})[
    ["avg score", "reached flag", "moves to flag", "avg moves outside"]]

> **[Instructor: what to land.]** With outside moves free, every new square outside is pure profit, even at
> the small default reward. What matters is whether there's a loop that earns points, not how big the reward
> is. **[Not yet checked at 20,000 attempts; at full scale, 2 of 3 training runs found the loop.]**

## Step 7: Combining skills nobody taught

Now the article's hard maze: the flag is walled off completely. No training maze ever required what it takes
to solve this one. To get there, the agent has to leave the maze, walk around the outside, and break back in
through a weak spot.

In [ ]:
mz.show_maze(mz.scenario("enclosed"))

**Can your default agent do it? Out of 500 tries, how many times do you think it will reach the flag?**

In [ ]:
# Out of 500 tries on the walled-off maze: how many left, broke back in from outside, and reached the flag?
mz.breakout_stages(default_run)

Our 20,000-attempt agent almost never manages it. It has learned to leave and come back, and it has learned
that some walls give way, but it hasn't put the two together yet. That only shows up with much more training.

The cell below trains the same default agent at the article's full scale: 126,000 attempts, about 15 minutes
on a T4. Start it and keep going with the discussion while it runs.

In [ ]:
# The default agent at full scale (about 15 minutes on a T4)
full_run = mz.train(rewards=default_rewards, episodes=126_000,
                    checkpoint_at=(1_000, 5_000, 20_000, 60_000))

# If short on time, load one trained the same way instead:
# full_run = mz.load_pretrained("base")

In [ ]:
display(mz.playback(full_run, maze="enclosed", n_agents=10, seed=0))

# How the combination shows up over training
compare({f"after {stage:,} attempts": mz.breakout_stages(full_run.agent_at(stage))
         for stage in sorted(full_run.checkpoints)})

> **[Instructor: what to land.]** Nobody trained "go around and break back in." The agent combined two
> skills it learned separately, because the +5 for the flag outweighs every small penalty along the way.
> This is the Hugging Face story from the article. And it doesn't happen every time: training the same way
> again can give an agent that does it 16% of the time or 75%.

## Step 8: Discussion

One more detail about how these agents were trained. I followed a rule from the article: if an attempt ends
with the agent **outside** the maze, we throw it away and don't learn from it. It's a safety rule, meant to
stop the agent from learning to escape.

In [ ]:
# How often did each agent's attempts end outside the maze (the ones the rule throws away)?
compare({"default": mz.evaluate(default_run, mazes=new_mazes),
         "big exploration reward": mz.evaluate(explore_run, mazes=new_mazes),
         "outside free": mz.evaluate(free_run, mazes=new_mazes)})[
    ["went around the maze", "avg moves outside"]]

**The hacking agents spend lots of time outside, but almost never *end* there. Why didn't the safety rule
catch them?**

**Who is responsible when an agent finds a loophole like this: the agent, the people who picked the rewards,
or the people who wrote the safety rule?**

**If you were building a sandbox for a real AI agent, what would you check besides the final outcome?**